## Behavior Raster Generator

This code is designed to take individual .csv files generated from ELAN, pre-process them and plot behaviors as individual subject and group raster plots. Created by NK 2026-03-19

Import relevant packages for analysis 

In [17]:
import os
import re
import csv
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# User input variables
CONFIG = {
    # define path for where raw .csv files for visualization can be found
    "analysis_root": "/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test",
    # define path for group assignments .csv
    "group_assignments_csv": "/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/GroupAssignments.csv",
    # define the behaviors that should be present in the data
    "allowed_behaviors": ["allogrooming", "allolicking", "selfgrooming", "self-licking", "huddling"],
    # define which behaviors to visualize
    "behaviors_to_visualize": ["allogrooming", "allolicking", "selfgrooming"],
    # define behavior to filter for bout duration
    "behavior_to_filter": "huddling",
    # define duration threshold (in seconds) for behavior to be considered
    "duration_threshold_seconds": 3,
    # define session duration (in seconds)
    "session_duration_seconds": 1800,
    # define gap threshold (in seconds) for combining bouts
    "bout_gap_threshold_seconds": 2.0,
    # define colors for each behavior to visualize
    "behavior_colors": {
        "allogrooming": "#49306B",
        "allolicking": "#D7BCE8",
        "selfgrooming": "#93E1D8",
        "self-licking": "#191308",
        "huddling": "#8884FF",
    },
}

CONFIG["analysis_root"] = os.path.abspath(CONFIG["analysis_root"])
CONFIG["preprocessed_folder"] = os.path.join(CONFIG["analysis_root"], "pre-processed")

analysis_root = CONFIG["analysis_root"]
preprocessed_folder = os.path.abspath(CONFIG["preprocessed_folder"])

# Compatibility aliases for downstream cells
NOTEBOOK_CONFIG = dict(CONFIG)
POST_RENAME_CONFIG = dict(CONFIG)

print("Configuration loaded.")
print("analysis_root:", analysis_root)
print("preprocessed_folder:", preprocessed_folder)

Configuration loaded.
analysis_root: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test
preprocessed_folder: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed


Convert all string to floats in .csv files for downstream analysis and filter out behaviors with durations less than the duration threshold, flag any files that contain behaviors outside the user defined behaviors 

In [18]:
# ===== USER VARIABLES =====
folder_path = os.path.abspath(globals().get('analysis_root', CONFIG["analysis_root"]))
preprocessed_folder = os.path.abspath(globals().get('preprocessed_folder', CONFIG["preprocessed_folder"]))
os.makedirs(preprocessed_folder, exist_ok=True)
# ==========================

canonical_behavior_map = {
    "allogrooming": "allogrooming",
    "allolicking": "allolicking",
    "selfgrooming": "selfgrooming",
    "self-grooming": "selfgrooming",
    "self-licking": "self-licking",
    "huddling": "huddling",
}

def normalize_behavior_label(raw_behavior):
    cleaned = raw_behavior.strip().replace('"', '').replace('_', '-')
    lowered = cleaned.lower()
    return canonical_behavior_map.get(lowered, cleaned)

def parse_row(row):
    cleaned_row = [str(value).strip().replace('"', '') for value in row]
    cleaned_row = cleaned_row[:-1] if cleaned_row and cleaned_row[-1] == '' else cleaned_row

    if len(cleaned_row) == 1:
        cleaned_row = [part.strip() for part in re.split(r'\t+|,', cleaned_row[0])]
        cleaned_row = cleaned_row[:-1] if cleaned_row and cleaned_row[-1] == '' else cleaned_row

    if not any(cleaned_row):
        return None

    behavior = normalize_behavior_label(cleaned_row[0])
    numeric_values = []

    for value in cleaned_row[1:]:
        if value == '':
            continue
        try:
            numeric_values.append(float(value))
        except ValueError:
            pass

    if not numeric_values:
        return [behavior]

    return [behavior, ''] + numeric_values

def write_rows_with_retries(path, rows, max_retries=5, sleep_seconds=1):
    for attempt in range(max_retries):
        try:
            with open(path, "w", newline="") as out_f:
                writer = csv.writer(out_f)
                writer.writerows(rows)
            return True
        except BlockingIOError:
            if attempt < max_retries - 1:
                print(
                    f"BlockingIOError for {os.path.basename(path)}, retrying in {sleep_seconds} second... "
                    f"(attempt {attempt + 1}/{max_retries})"
                )
                time.sleep(sleep_seconds)
            else:
                print(f"Skipping locked pre-processed file after {max_retries} attempts: {path}")
    return False

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    input_path = os.path.join(folder_path, filename)
    output_path = os.path.join(preprocessed_folder, filename)

    fixed_rows = []

    with open(input_path, "r", newline="") as f:
        sample = f.read(2048)
        f.seek(0)

        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t")
        except csv.Error:
            dialect = csv.excel
            dialect.delimiter = '\t' if '\t' in sample else ','

        reader = csv.reader(f, dialect)

        for row in reader:
            parsed_row = parse_row(row)
            if parsed_row is not None:
                fixed_rows.append(parsed_row)

    if write_rows_with_retries(output_path, fixed_rows):
        print(f"Pre-processed CSV saved: {output_path}")

Pre-processed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/C1-L_X_postMEL_NK+KLT.csv
Pre-processed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/C1-R_RL_postMEL_PRB+KLT.csv
Pre-processed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/C2-L_X_postMEL_MW+KLT.csv
Pre-processed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/C2-R_RL_postMEL_NK+KLT.csv


In [19]:
folder_path = os.path.abspath(globals().get('preprocessed_folder', CONFIG["preprocessed_folder"]))
allowed_behaviors = set(CONFIG["allowed_behaviors"])

unexpected_behaviors = set()

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, filename)
    file_invalid_behaviors = set()

    with open(file_path, newline='') as f:
        sample = f.read(2048)
        f.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t")
        except csv.Error:
            dialect = csv.get_dialect('excel')
            dialect.delimiter = '\t' if '\t' in sample else ','
        reader = csv.reader(f, dialect)

        for row in reader:
            if not row:
                continue
            behavior = row[0]
            behavior_clean = behavior.strip().replace('"', '').replace('_', '-').lower()
            if behavior_clean and behavior_clean not in allowed_behaviors:
                file_invalid_behaviors.add(behavior_clean)
                unexpected_behaviors.add(behavior_clean)

    if file_invalid_behaviors:
        print(f"{filename} contains unexpected behaviors (rows may have issues): {file_invalid_behaviors}")

print("\nSummary of all unexpected behaviors across files:")
print(unexpected_behaviors)


Summary of all unexpected behaviors across files:
set()


In [20]:
# ===== USER VARIABLES =====
folder_path = os.path.abspath(globals().get('preprocessed_folder', CONFIG["preprocessed_folder"]))
behavior_to_filter = CONFIG["behavior_to_filter"]
duration_threshold = CONFIG["duration_threshold_seconds"]
# ==========================

def robust_read_csv(path):
    for sep, kwargs in [("\t", {}), (",", {}), (r"\s+", {"engine": "python"})]:
        try:
            df = pd.read_csv(path, sep=sep, header=None, **kwargs)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    rows = [re.split(r"\t+|,|\s{2,}|\s+", line) for line in lines]
    return pd.DataFrame(rows)

skipped_locked_files = []

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    input_path = os.path.join(folder_path, filename)
    output_path = os.path.join(folder_path, filename)

    df = robust_read_csv(input_path)

    if df.shape[1] > 4:
        dur_col = 4
    else:
        numeric_counts = []
        for col in df.columns:
            numeric_counts.append((col, pd.to_numeric(df[col], errors="coerce").notna().sum()))
        dur_col = max(numeric_counts, key=lambda x: x[1])[0]

    df[dur_col] = pd.to_numeric(df[dur_col], errors="coerce")
    filtered_df = df[
        ~(
            (df[0].astype(str).str.strip().str.lower() == behavior_to_filter)
            & (df[dur_col] < duration_threshold)
        )
    ]

    max_retries = 5
    saved_ok = False
    for attempt in range(max_retries):
        try:
            filtered_df.to_csv(output_path, sep=",", header=False, index=False)
            saved_ok = True
            break
        except BlockingIOError:
            if attempt < max_retries - 1:
                print(
                    f"BlockingIOError for {filename}, retrying in 1 second... "
                    f"(attempt {attempt + 1}/{max_retries})"
                )
                time.sleep(1)
            else:
                skipped_locked_files.append(filename)
                print(f"Skipping locked file after {max_retries} attempts: {filename}")

    if not saved_ok:
        continue

    removed_rows = len(df) - len(filtered_df)
    print(f"{filename}: removed {removed_rows} rows | saved to pre-processed folder")

print("\nFiltering complete. Files saved to:", folder_path)
if skipped_locked_files:
    print("Skipped locked files:", skipped_locked_files)

C1-L_X_postMEL_NK+KLT.csv: removed 2 rows | saved to pre-processed folder
C1-R_RL_postMEL_PRB+KLT.csv: removed 8 rows | saved to pre-processed folder
C2-L_X_postMEL_MW+KLT.csv: removed 0 rows | saved to pre-processed folder
C2-R_RL_postMEL_NK+KLT.csv: removed 2 rows | saved to pre-processed folder

Filtering complete. Files saved to: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed


Create plots saved as png for each individual .csv file 

In [21]:
plt.rcParams['font.family'] = 'Arial'

# ===== USER VARIABLES =====
plot_config = globals().get('CONFIG', NOTEBOOK_CONFIG)
input_folder = os.path.abspath(globals().get('preprocessed_folder', plot_config["preprocessed_folder"]))
plot_folder = os.path.join(input_folder, 'plots')
os.makedirs(plot_folder, exist_ok=True)
session_duration = plot_config["session_duration_seconds"]
behaviors_to_visualize = list(plot_config["behaviors_to_visualize"])
bout_gap_threshold = plot_config["bout_gap_threshold_seconds"]
static_behavior_colors = dict(plot_config.get("behavior_colors", {}))
# ==========================


# robust reader
def robust_read_csv(path):
    for sep, kwargs in [("\t", {}), (",", {}), (r"\s+", {"engine": "python"})]:
        try:
            df = pd.read_csv(path, sep=sep, header=None, **kwargs)
            if df.shape[1] > 1:
                return df, sep
        except Exception:
            pass

    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    rows = [re.split(r"\t+|,|\s{2,}|\s+", line) for line in lines]
    return pd.DataFrame(rows), "\t"


def normalize_behavior(value):
    raw = str(value).strip().lower().replace('_', '-')
    key = re.sub(r'[^a-z-]', '', raw)
    aliases = {
        'selflicking': 'self-licking',
        'self-licking': 'self-licking',
        'selfgrooming': 'selfgrooming',
        'allogrooming': 'allogrooming',
        'allolicking': 'allolicking',
        'huddling': 'huddling'
    }
    return aliases.get(key, key)


def parse_events(df):
    behaviors = []
    starts = []
    ends = []

    for _, row in df.iterrows():
        behavior = normalize_behavior(row.iloc[0])
        nums = []
        for val in row.iloc[1:]:
            num = pd.to_numeric(val, errors='coerce')
            if pd.notna(num):
                nums.append(float(num))

        if len(nums) >= 2:
            start_time = nums[0]
            end_time = nums[1]
            if end_time >= start_time:
                behaviors.append(behavior)
                starts.append(start_time)
                ends.append(end_time)

    return pd.DataFrame({'behavior': behaviors, 'start_time': starts, 'end_time': ends})


def count_bouts(df, behavior, gap_threshold=2.0):
    if df.empty:
        return 0

    behavior_df = df[df['behavior'] == behavior].sort_values(by=['start_time', 'end_time']).reset_index(drop=True)
    if behavior_df.empty:
        return 0

    bout_count = 1
    current_end = behavior_df.loc[0, 'end_time']

    for i in range(1, len(behavior_df)):
        next_start = behavior_df.loc[i, 'start_time']
        next_end = behavior_df.loc[i, 'end_time']
        gap = next_start - current_end

        if gap >= gap_threshold:
            bout_count += 1
            current_end = next_end
        else:
            current_end = max(current_end, next_end)

    return bout_count

def get_color_for_behavior(behavior, known_colors, all_unique_behaviors):
    """Get color for a behavior, using predefined colors when available"""
    if behavior in known_colors:
        return known_colors[behavior]
    unknown_behaviors = [b for b in sorted(all_unique_behaviors) if b not in known_colors]
    if unknown_behaviors:
        extra_colors = plt.cm.tab20(np.linspace(0, 1, len(unknown_behaviors)))
        idx = unknown_behaviors.index(behavior)
        return extra_colors[idx]
    return '#000000'

# Collect all unique behaviors first to ensure consistency
all_behaviors_set = set()
for filename in sorted(os.listdir(input_folder)):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_folder, filename)
        df_temp, _ = robust_read_csv(filepath)
        if not df_temp.empty:
            events_temp = parse_events(df_temp)
            if not events_temp.empty:
                all_behaviors_set.update(events_temp['behavior'].unique())

# Create consistent color mapping for all behaviors
behavior_colors = {}
for behavior in all_behaviors_set:
    behavior_colors[behavior] = get_color_for_behavior(behavior, static_behavior_colors, all_behaviors_set)

# List to store processed data for combined plot
processed_data = []

# Process each CSV file in the pre-processed folder
for filename in sorted(os.listdir(input_folder)):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(input_folder, filename)
    df_raw, _ = robust_read_csv(filepath)

    if df_raw.empty:
        print(f"Skipping empty file: {filename}")
        continue

    df = parse_events(df_raw)

    if df.empty:
        print(f"No valid behavior data in: {filename}")
        continue

    # Filter to only include specified behaviors
    df = df[df['behavior'].isin(behaviors_to_visualize)]

    # Get unique behaviors in this file
    unique_behaviors = sorted(df['behavior'].unique())

    total_bouts = {
        behavior: count_bouts(df, behavior, bout_gap_threshold)
        for behavior in behaviors_to_visualize
    }
    bouts_text = (
        f"Bouts: allogrooming={total_bouts['allogrooming']}, "
        f"allolicking={total_bouts['allolicking']}, "
        f"selfgrooming={total_bouts['selfgrooming']}"
    )

    title = os.path.splitext(filename)[0]
    title = re.sub(r'_postMEL.*', '', title)
    processed_data.append((title, df, unique_behaviors, total_bouts))

    fig, ax = plt.subplots(figsize=(16, 3))

    if df.empty:
        ax.text(0.5, 0.5, 'No behaviors to display', transform=ax.transAxes, ha='center', va='center', fontsize=16)
    else:
        for _, row in df.iterrows():
            behavior = row['behavior']
            start = row['start_time']
            end = row['end_time']
            rect = mpatches.Rectangle((start, 0), end - start, 1,
                                     facecolor=behavior_colors.get(behavior, '#000000'),
                                     edgecolor='none')
            ax.add_patch(rect)

    ax.set_xlim(0, session_duration)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Time (minutes)', fontsize=30)

    xticks = np.arange(0, session_duration + 1, 300)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f'{int(x//60)}' for x in xticks])

    ax.set_yticks([])
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(3)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(3)
    ax.spines['right'].set_visible(True)
    ax.spines['right'].set_color('black')
    ax.spines['right'].set_linewidth(3)
    ax.spines['top'].set_visible(True)
    ax.spines['top'].set_color('black')
    ax.spines['top'].set_linewidth(3)

    ax.text(-0.1, 0.5, title, transform=ax.transAxes, fontsize=30, fontweight='bold', ha='right', va='center')

    ax.text(
        -0.1,
        0.28,
        bouts_text,
        transform=ax.transAxes,
        fontsize=20,
        fontweight='bold',
        ha='right',
        va='center'
    )

    if not df.empty:
        legend_patches = [mpatches.Patch(facecolor=behavior_colors[behavior],
                                        label=behavior) for behavior in unique_behaviors]
        ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=30)

    plt.tight_layout()

    output_filename = os.path.splitext(filename)[0] + '.png'
    output_path = os.path.join(plot_folder, output_filename)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved raster plot: {output_path}")

    plt.close()

if processed_data:
    nrows = len(processed_data)
    fig, axes = plt.subplots(nrows=nrows, figsize=(24, 3 * nrows))

    if nrows == 1:
        axes = [axes]

    for i, (title, df, unique_behaviors, total_bouts) in enumerate(processed_data):
        ax = axes[i]

        for _, row in df.iterrows():
            behavior = row['behavior']
            start = row['start_time']
            end = row['end_time']
            rect = mpatches.Rectangle((start, 0), end - start, 1,
                                     facecolor=behavior_colors.get(behavior, '#000000'),
                                     edgecolor='none')
            ax.add_patch(rect)

        ax.set_xlim(0, session_duration)
        ax.set_ylim(0, 1)

        if i == nrows - 1:
            ax.set_xlabel('Time (minutes)', fontsize=30)
            xticks = np.arange(0, session_duration + 1, 300)
            ax.set_xticks(xticks)
            ax.set_xticklabels([f'{int(x//60)}' for x in xticks])
        else:
            ax.set_xticks([])

        ax.set_yticks([])

        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color('black')
        ax.spines['left'].set_linewidth(3)
        ax.spines['bottom'].set_visible(i == nrows - 1)
        ax.spines['bottom'].set_color('black')
        ax.spines['bottom'].set_linewidth(3)
        ax.spines['right'].set_visible(True)
        ax.spines['right'].set_color('black')
        ax.spines['right'].set_linewidth(3)
        ax.spines['top'].set_visible(i == 0)
        ax.spines['top'].set_color('black')
        ax.spines['top'].set_linewidth(3)

        ax.text(-0.1, 0.5, title, transform=ax.transAxes, fontsize=30, fontweight='bold', ha='right', va='center')

        bouts_text = (
            f"Bouts: allogrooming={total_bouts['allogrooming']}, "
            f"allolicking={total_bouts['allolicking']}, "
            f"selfgrooming={total_bouts['selfgrooming']}"
        )
        ax.text(
            -0.1,
            0.28,
            bouts_text,
            transform=ax.transAxes,
            fontsize=20,
            fontweight='bold',
            ha='right',
            va='center'
        )

        legend_patches = [mpatches.Patch(facecolor=behavior_colors[behavior], label=behavior) for behavior in unique_behaviors]
        ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=30)

    plt.tight_layout()
    combined_path = os.path.join(plot_folder, 'combined_raster_plots.png')
    plt.savefig(combined_path, dpi=300, bbox_inches='tight')
    print(f"Saved combined raster plot: {combined_path}")
    plt.close()

print("\nRaster plots complete. All PNG files saved to:", plot_folder)

Saved raster plot: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/C1-L_X_postMEL_NK+KLT.png
Saved raster plot: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/C1-R_RL_postMEL_PRB+KLT.png
Saved raster plot: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/C2-L_X_postMEL_MW+KLT.png
Saved raster plot: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/C2-R_RL_postMEL_NK+KLT.png
Saved combined raster plot: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/combined_raster_plots.png

Raster plots complete. All PNG files saved to: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots


Create plots for within subject comparison of different conditions, one png is generated per animal with plots for both conditions, combined png includes all animals with each condition plotted

Note: this is for experiments in which the SToP is run 2x, ie one round the animal recieved drugs, and the next round the animal recieved saline

In [11]:
# Load CSV containing information regarding treatment for each subject and create plot with rasters that compare the drug vs saline condition for each animal

import pandas as pd
import os
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plot_config = globals().get('CONFIG', NOTEBOOK_CONFIG)
input_folder = os.path.abspath(globals().get('preprocessed_folder', plot_config["preprocessed_folder"]))
pair_plot_folder = os.path.join(input_folder, 'plots', 'withinSubject')
os.makedirs(pair_plot_folder, exist_ok=True)
session_duration = plot_config["session_duration_seconds"]
bout_gap_threshold = plot_config["bout_gap_threshold_seconds"]
behaviors_to_visualize = list(plot_config["behaviors_to_visualize"])
static_behavior_colors = dict(plot_config.get("behavior_colors", {}))

def robust_read_csv(path):
    for sep, kwargs in [("\t", {}), (",", {}), (r"\s+", {"engine": "python"})]:
        try:
            df = pd.read_csv(path, sep=sep, header=None, **kwargs)
            if df.shape[1] > 1:
                return df, sep
        except Exception:
            pass
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    rows = [re.split(r"\t+|,|\s{2,}|\s+", line) for line in lines]
    df = pd.DataFrame(rows)
    return df, "\t"


def normalize_behavior(value):
    raw = str(value).strip().lower().replace('_', '-')
    key = re.sub(r'[^a-z-]', '', raw)
    aliases = {
        'selflicking': 'self-licking',
        'self-licking': 'self-licking',
        'selfgrooming': 'selfgrooming',
        'allogrooming': 'allogrooming',
        'allolicking': 'allolicking',
        'huddling': 'huddling'
    }
    return aliases.get(key, key)


def parse_events(df):
    behaviors = []
    starts = []
    ends = []

    for _, row in df.iterrows():
        behavior = normalize_behavior(row.iloc[0])
        nums = []
        for val in row.iloc[1:]:
            num = pd.to_numeric(val, errors='coerce')
            if pd.notna(num):
                nums.append(float(num))

        if len(nums) >= 2:
            start_time = nums[0]
            end_time = nums[1]
            if end_time >= start_time:
                behaviors.append(behavior)
                starts.append(start_time)
                ends.append(end_time)

    return pd.DataFrame({'behavior': behaviors, 'start_time': starts, 'end_time': ends})

assignments_path = plot_config["group_assignments_csv"]
assignments = pd.read_csv(assignments_path)
assignments.columns = assignments.columns.str.strip()

cohort_to_pairs = {}
for _, row in assignments.iterrows():
    cohort = row['Cohort']
    round_num = row['Round']
    pair = row['Pair']
    treatment = row['Treatment']
    if cohort not in cohort_to_pairs:
        cohort_to_pairs[cohort] = {}
    if pair not in cohort_to_pairs[cohort]:
        cohort_to_pairs[cohort][pair] = {}
    cohort_to_pairs[cohort][pair][round_num] = treatment

def get_color_for_behavior(behavior, known_colors, all_unique_behaviors):
    if behavior in known_colors:
        return known_colors[behavior]
    unknown_behaviors = [b for b in sorted(all_unique_behaviors) if b not in known_colors]
    if unknown_behaviors:
        extra_colors = plt.cm.tab20(np.linspace(0, 1, len(unknown_behaviors)))
        idx = unknown_behaviors.index(behavior)
        return extra_colors[idx]
    return '#000000'

def count_bouts(df, behavior, gap_threshold=2.0):
    if df.empty:
        return 0
    behavior_df = df[df['behavior'] == behavior].sort_values(by=['start_time', 'end_time']).reset_index(drop=True)
    if behavior_df.empty:
        return 0

    bout_count = 1
    current_end = behavior_df.loc[0, 'end_time']

    for k in range(1, len(behavior_df)):
        next_start = behavior_df.loc[k, 'start_time']
        next_end = behavior_df.loc[k, 'end_time']
        gap = next_start - current_end

        if gap >= gap_threshold:
            bout_count += 1
            current_end = next_end
        else:
            current_end = max(current_end, next_end)

    return bout_count

all_behaviors_set = set()
for filename in sorted(os.listdir(input_folder)):
    if filename.endswith('.csv'):
        filepath = os.path.join(input_folder, filename)
        df_temp, _ = robust_read_csv(filepath)
        if not df_temp.empty:
            events_temp = parse_events(df_temp)
            if not events_temp.empty:
                all_behaviors_set.update(events_temp['behavior'].unique())

behavior_colors = {}
for behavior in all_behaviors_set:
    behavior_colors[behavior] = get_color_for_behavior(behavior, static_behavior_colors, all_behaviors_set)

pair_data_list = []

for cohort, pairs in cohort_to_pairs.items():
    for pair, rounds in pairs.items():
        if 1 not in rounds or 2 not in rounds:
            print(f"Cohort {cohort} Pair {pair} does not have both rounds, skipping")
            continue

        matching_files1 = [f for f in os.listdir(input_folder) if f.startswith(f"Cohort{cohort}_Round1_{pair}") and f.endswith('.csv')]
        matching_files2 = [f for f in os.listdir(input_folder) if f.startswith(f"Cohort{cohort}_Round2_{pair}") and f.endswith('.csv')]

        if not matching_files1 or not matching_files2:
            print(f"Files not found for Cohort {cohort} Pair {pair}: Round1 {matching_files1}, Round2 {matching_files2}")
            continue

        file1 = os.path.join(input_folder, matching_files1[0])
        file2 = os.path.join(input_folder, matching_files2[0])

        df1_raw, _ = robust_read_csv(file1)
        df2_raw, _ = robust_read_csv(file2)
        df1 = parse_events(df1_raw)
        df2 = parse_events(df2_raw)

        saline_round = 1 if str(rounds[1]).strip().upper() == 'SALINE' else 2
        drug_round = 3 - saline_round

        data_list = []
        for round_num, treat in [(saline_round, 'SALINE'), (drug_round, 'DRUG')]:
            df = df1.copy() if round_num == 1 else df2.copy()
            if not df.empty:
                df = df[df['behavior'].isin(behaviors_to_visualize)]
            data_list.append((df, f"{pair} Round {round_num} ({treat})"))

        pair_data_list.append((f"Cohort {cohort} Pair {pair}", data_list))

        fig, axes = plt.subplots(1, 2, figsize=(24, 5))

        for i, data in enumerate(data_list):
            ax = axes[i]
            if data[0].empty:
                ax.text(0.5, 0.5, 'No behaviors to display', transform=ax.transAxes, ha='center', va='center', fontsize=16)
                ax.set_title(data[1], fontsize=30, fontweight='bold')
                continue

            df, label = data
            for _, row in df.iterrows():
                behavior = row['behavior']
                start = row['start_time']
                end = row['end_time']
                ax.add_patch(mpatches.Rectangle((start, 0), end - start, 1, facecolor=behavior_colors.get(behavior, '#000000'), edgecolor='none'))

            ax.set_xlim(0, session_duration)
            ax.set_ylim(0, 1)
            ax.set_xlabel('Time (s)', fontsize=30)
            ax.set_title(label, fontsize=30, fontweight='bold')
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(3)

        plt.tight_layout()
        legend_patches = [mpatches.Patch(facecolor=behavior_colors.get(behavior, '#000000'), label=behavior) for behavior in behaviors_to_visualize]
        fig.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=30)
        output_path = os.path.join(pair_plot_folder, f"Cohort{cohort}_{pair}_drug_vs_saline.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved raster for Cohort {cohort} Pair {pair}: {output_path}")

if pair_data_list:
    num_plots = len(pair_data_list)
    fig, axes = plt.subplots(num_plots, 2, figsize=(48, 5 * num_plots))
    if num_plots == 1:
        axes = [axes]

    for i, (title, data_list) in enumerate(pair_data_list):
        parts = title.split()
        cohort = parts[1]
        pair = parts[3]
        subject = f"{cohort}_{pair}"

        saline_bouts = {behavior: count_bouts(data_list[0][0], behavior, bout_gap_threshold) for behavior in behaviors_to_visualize}
        drug_bouts = {behavior: count_bouts(data_list[1][0], behavior, bout_gap_threshold) for behavior in behaviors_to_visualize}

        for j, data in enumerate(data_list):
            if num_plots > 1:
                ax = axes[i, j]
            else:
                ax = axes[j]

            if data[0].empty:
                ax.text(0.5, 0.5, 'No behaviors to display', transform=ax.transAxes, ha='center', va='center', fontsize=16)
                continue

            df, label = data
            for _, row in df.iterrows():
                behavior = row['behavior']
                start = row['start_time']
                end = row['end_time']
                ax.add_patch(mpatches.Rectangle((start, 0), end - start, 1, facecolor=behavior_colors.get(behavior, '#000000'), edgecolor='none'))

            ax.set_xlim(0, session_duration)
            ax.set_ylim(0, 1)
            if i == num_plots - 1:
                ax.set_xlabel('Time (s)', fontsize=60)
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(3)

        if num_plots > 1:
            axes[i, 0].text(-0.15, 0.5, subject, transform=axes[i, 0].transAxes, fontsize=60, ha='right', va='center', fontweight='bold')
            axes[i, 0].text(-0.15, 0.35, f"Saline: allogrooming={saline_bouts['allogrooming']}, allolicking={saline_bouts['allolicking']}, selfgrooming={saline_bouts['selfgrooming']}", transform=axes[i, 0].transAxes, fontsize=35, ha='right', va='center')
            axes[i, 0].text(-0.15, 0.25, f"Drug: allogrooming={drug_bouts['allogrooming']}, allolicking={drug_bouts['allolicking']}, selfgrooming={drug_bouts['selfgrooming']}", transform=axes[i, 0].transAxes, fontsize=35, ha='right', va='center')
        else:
            axes[0].text(-0.15, 0.5, subject, transform=axes[0].transAxes, fontsize=60, ha='right', va='center', fontweight='bold')
            axes[0].text(-0.15, 0.35, f"Saline: allogrooming={saline_bouts['allogrooming']}, allolicking={saline_bouts['allolicking']}, selfgrooming={saline_bouts['selfgrooming']}", transform=axes[0].transAxes, fontsize=35, ha='right', va='center')
            axes[0].text(-0.15, 0.25, f"Drug: allogrooming={drug_bouts['allogrooming']}, allolicking={drug_bouts['allolicking']}, selfgrooming={drug_bouts['selfgrooming']}", transform=axes[0].transAxes, fontsize=35, ha='right', va='center')

    plt.tight_layout()
    if num_plots > 1:
        axes[0, 0].text(0.5, 1.02, 'Saline', transform=axes[0, 0].transAxes, ha='center', va='bottom', fontsize=60, fontweight='bold')
        axes[0, 1].text(0.5, 1.02, 'Drug', transform=axes[0, 1].transAxes, ha='center', va='bottom', fontsize=60, fontweight='bold')
    else:
        axes[0].text(0.5, 1.02, 'Saline', transform=axes[0].transAxes, ha='center', va='bottom', fontsize=60, fontweight='bold')
        axes[1].text(0.5, 1.02, 'Drug', transform=axes[1].transAxes, ha='center', va='bottom', fontsize=60, fontweight='bold')

    legend_patches = [mpatches.Patch(facecolor=behavior_colors.get(behavior, '#000000'), label=behavior) for behavior in behaviors_to_visualize]
    fig.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=60)
    combined_path = os.path.join(pair_plot_folder, 'all_pairs_drug_vs_saline.png')
    plt.savefig(combined_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved combined raster for all pairs: {combined_path}")

print('All pair rasters created.')

Saved raster for Cohort 1 Pair C1-L_X: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/pre-processed/plots/withinSubject/Cohort1_C1-L_X_drug_vs_saline.png
Saved raster for Cohort 1 Pair C1-R_RL: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/pre-processed/plots/withinSubject/Cohort1_C1-R_RL_drug_vs_saline.png
Saved raster for Cohort 1 Pair C2-L_X: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/pre-processed/plots/withinSubject/Cohort1_C2-L_X_drug_vs_saline.png
Saved raster for Cohort 1 Pair C2-R_RL: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/pre-processed/plots/withinSubject/Cohort1_C2-R_RL_drug_vs_saline.png
Saved raster for Cohort 1 Pair C3-L_X: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/pre-processed/plots/withinSubject/Cohort1_C3-L_X_drug_vs_saline.png
Saved raster for Cohort 1 Pair C3-R_RL: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI

Group-based raster plots from GroupAssignments.csv

Generate subject-level rasters labeled with group assignment from GroupAssignments.csv (columns: Subject, Group), plus a combined figure that places CONTROL subjects next to EXPERIMENTAL subjects.

In [25]:
# Group-based subject comparison using GroupAssignments.csv (Subject + Group/Treatment)
plot_config = globals().get('CONFIG', NOTEBOOK_CONFIG)
input_folder = os.path.abspath(globals().get('preprocessed_folder', plot_config["preprocessed_folder"]))
group_plot_folder = os.path.join(input_folder, 'plots', 'byGroup')
os.makedirs(group_plot_folder, exist_ok=True)

session_duration = plot_config["session_duration_seconds"]
behaviors_to_visualize = list(plot_config["behaviors_to_visualize"])
static_behavior_colors = dict(plot_config.get("behavior_colors", {}))

assignments_path = plot_config["group_assignments_csv"]
assignments = pd.read_csv(assignments_path)
assignments.columns = assignments.columns.str.strip()

col_lookup = {c.strip().lower(): c for c in assignments.columns}
subject_col = col_lookup.get('subject')
group_col = col_lookup.get('group') or col_lookup.get('treatment')
if subject_col is None or group_col is None:
    raise ValueError(
        f"GroupAssignments.csv must contain 'Subject' and either 'Group' or 'Treatment'. Found: {assignments.columns.tolist()}"
    )

def normalize_key(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())

def canonical_group_label(value):
    raw = str(value).strip().upper()
    control_aliases = {'CONTROL', 'CTRL', 'SALINE', 'VEHICLE'}
    experimental_aliases = {'EXPERIMENTAL', 'EXP', 'DRUG', 'TREATMENT'}
    if raw in control_aliases:
        return 'CONTROL'
    if raw in experimental_aliases:
        return 'EXPERIMENTAL'
    return raw

subject_to_group = {}
for _, row in assignments.iterrows():
    subject_key = normalize_key(row[subject_col])
    group_value = canonical_group_label(row[group_col])
    if subject_key:
        subject_to_group[subject_key] = group_value

def robust_read_csv(path):
    for sep, kwargs in [("\t", {}), (",", {}), (r"\s+", {"engine": "python"})]:
        try:
            df = pd.read_csv(path, sep=sep, header=None, **kwargs)
            if df.shape[1] > 1:
                return df, sep
        except Exception:
            pass
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    rows = [re.split(r"\t+|,|\s{2,}|\s+", line) for line in lines]
    return pd.DataFrame(rows), "\t"

def normalize_behavior(value):
    raw = str(value).strip().lower().replace('_', '-')
    key = re.sub(r'[^a-z-]', '', raw)
    aliases = {
        'selflicking': 'self-licking',
        'self-licking': 'self-licking',
        'selfgrooming': 'selfgrooming',
        'allogrooming': 'allogrooming',
        'allolicking': 'allolicking',
        'huddling': 'huddling'
    }
    return aliases.get(key, key)

def parse_events(df):
    behaviors = []
    starts = []
    ends = []
    for _, row in df.iterrows():
        behavior = normalize_behavior(row.iloc[0])
        nums = []
        for val in row.iloc[1:]:
            num = pd.to_numeric(val, errors='coerce')
            if pd.notna(num):
                nums.append(float(num))

        if len(nums) >= 2:
            start_time = nums[0]
            end_time = nums[1]
            if end_time >= start_time:
                behaviors.append(behavior)
                starts.append(start_time)
                ends.append(end_time)
    return pd.DataFrame({'behavior': behaviors, 'start_time': starts, 'end_time': ends})

def get_color_for_behavior(behavior, known_colors, all_unique_behaviors):
    if behavior in known_colors:
        return known_colors[behavior]
    unknown_behaviors = [b for b in sorted(all_unique_behaviors) if b not in known_colors]
    if unknown_behaviors:
        extra_colors = plt.cm.tab20(np.linspace(0, 1, len(unknown_behaviors)))
        idx = unknown_behaviors.index(behavior)
        return extra_colors[idx]
    return '#000000'

def extract_subject_candidates(base_name):
    candidates = [base_name]
    candidates.append(re.sub(r'_postMEL.*$', '', base_name, flags=re.IGNORECASE))
    candidates.append(re.sub(r'^Cohort\d+_Round\d+_', '', base_name, flags=re.IGNORECASE))
    candidates.append(
        re.sub(
            r'^Cohort\d+_Round\d+_',
            '',
            re.sub(r'_postMEL.*$', '', base_name, flags=re.IGNORECASE),
            flags=re.IGNORECASE
        )
    )
    unique = []
    for c in candidates:
        if c not in unique:
            unique.append(c)
    return unique

def resolve_group_from_filename(filename):
    base = os.path.splitext(filename)[0]
    for candidate in extract_subject_candidates(base):
        key = normalize_key(candidate)
        if key in subject_to_group:
            return subject_to_group[key], candidate
    return None, base

def draw_subject_raster(ax, df, subject_label, behavior_colors, label_side='left'):
    if df.empty:
        ax.text(0.5, 0.5, 'No behaviors to display', transform=ax.transAxes, ha='center', va='center', fontsize=14)
    else:
        for _, row in df.iterrows():
            ax.add_patch(
                mpatches.Rectangle(
                    (row['start_time'], 0),
                    row['end_time'] - row['start_time'],
                    1,
                    facecolor=behavior_colors.get(row['behavior'], '#000000'),
                    edgecolor='none'
                )
            )

    ax.set_xlim(0, session_duration)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(np.arange(0, session_duration + 1, 300))
    ax.set_xticklabels([f"{int(x // 60)}" for x in np.arange(0, session_duration + 1, 300)])
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

    if label_side == 'right':
        x_pos = 1.03
        ha = 'left'
    else:
        x_pos = -0.03
        ha = 'right'

    ax.text(
        x_pos,
        0.5,
        subject_label,
        transform=ax.transAxes,
        ha=ha,
        va='center',
        fontsize=16,
        fontweight='bold'
    )

all_behaviors_set = set()
for filename in sorted(os.listdir(input_folder)):
    if not filename.endswith('.csv'):
        continue
    filepath = os.path.join(input_folder, filename)
    df_temp, _ = robust_read_csv(filepath)
    if df_temp.empty:
        continue
    events_temp = parse_events(df_temp)
    if not events_temp.empty:
        all_behaviors_set.update(events_temp['behavior'].unique())

behavior_colors = {
    behavior: get_color_for_behavior(behavior, static_behavior_colors, all_behaviors_set)
    for behavior in all_behaviors_set
}

subject_entries = []
unmatched_files = []

for filename in sorted(os.listdir(input_folder)):
    if not filename.endswith('.csv'):
        continue

    filepath = os.path.join(input_folder, filename)
    df_raw, _ = robust_read_csv(filepath)
    if df_raw.empty:
        continue

    df = parse_events(df_raw)
    if df.empty:
        continue

    df = df[df['behavior'].isin(behaviors_to_visualize)]
    group_value, matched_subject = resolve_group_from_filename(filename)
    if group_value is None:
        unmatched_files.append(filename)
        continue

    base_title = re.sub(r'_postMEL.*$', '', os.path.splitext(filename)[0], flags=re.IGNORECASE)
    display_title = f"{base_title} [{group_value}]"
    subject_entries.append({
        'filename': filename,
        'subject': matched_subject,
        'group': group_value,
        'title': display_title,
        'df': df
    })

if unmatched_files:
    print('Files with no Subject match in GroupAssignments.csv:')
    for fn in unmatched_files:
        print('-', fn)

for entry in subject_entries:
    fig, ax = plt.subplots(figsize=(16, 3))
    draw_subject_raster(ax, entry['df'], entry['title'], behavior_colors, label_side='left')
    ax.set_xlabel('Time (minutes)', fontsize=16)
    legend_behaviors = sorted(entry['df']['behavior'].unique())
    legend_patches = [mpatches.Patch(facecolor=behavior_colors[b], label=b) for b in legend_behaviors]
    if legend_patches:
        ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=12)
    plt.tight_layout()
    out_name = os.path.splitext(entry['filename'])[0] + '_by_group.png'
    out_path = os.path.join(group_plot_folder, out_name)
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved grouped subject raster: {out_path}")

control_entries = [e for e in subject_entries if e['group'] == 'CONTROL']
experimental_entries = [e for e in subject_entries if e['group'] == 'EXPERIMENTAL']
other_group_entries = [e for e in subject_entries if e['group'] not in {'CONTROL', 'EXPERIMENTAL'}]

if other_group_entries:
    print('Subjects with non-CONTROL/EXPERIMENTAL labels were not included in side-by-side plot:')
    for e in other_group_entries:
        print('-', e['title'])

max_rows = max(len(control_entries), len(experimental_entries))
if max_rows > 0:
    fig, axes = plt.subplots(max_rows, 2, figsize=(36, 4 * max_rows), squeeze=False)

    for i in range(max_rows):
        left_ax = axes[i, 0]
        right_ax = axes[i, 1]

        if i < len(control_entries):
            draw_subject_raster(
                left_ax,
                control_entries[i]['df'],
                control_entries[i]['title'],
                behavior_colors,
                label_side='left'
            )
            if i == max_rows - 1:
                left_ax.set_xlabel('Time (minutes)', fontsize=16)
            else:
                left_ax.set_xticklabels([])
        else:
            left_ax.axis('off')
            left_ax.text(0.5, 0.5, 'No CONTROL subject', ha='center', va='center', transform=left_ax.transAxes, fontsize=12)

        if i < len(experimental_entries):
            draw_subject_raster(
                right_ax,
                experimental_entries[i]['df'],
                experimental_entries[i]['title'],
                behavior_colors,
                label_side='right'
            )
            if i == max_rows - 1:
                right_ax.set_xlabel('Time (minutes)', fontsize=16)
            else:
                right_ax.set_xticklabels([])
        else:
            right_ax.axis('off')
            right_ax.text(0.5, 0.5, 'No EXPERIMENTAL subject', ha='center', va='center', transform=right_ax.transAxes, fontsize=12)

    axes[0, 0].text(0.5, 1.05, 'CONTROL', transform=axes[0, 0].transAxes, ha='center', va='bottom', fontsize=22, fontweight='bold')
    axes[0, 1].text(0.5, 1.05, 'EXPERIMENTAL', transform=axes[0, 1].transAxes, ha='center', va='bottom', fontsize=22, fontweight='bold')

    legend_patches = [mpatches.Patch(facecolor=behavior_colors.get(b, '#000000'), label=b) for b in behaviors_to_visualize]
    fig.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=14)
    plt.tight_layout()

    grouped_path = os.path.join(group_plot_folder, 'group_comparison_control_vs_experimental.png')
    plt.savefig(grouped_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved group comparison raster: {grouped_path}")
else:
    print('No matched subjects found for CONTROL/EXPERIMENTAL plotting.')

print('Group-based raster plotting complete.')

Saved grouped subject raster: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/byGroup/C1-L_X_postMEL_NK+KLT_by_group.png
Saved grouped subject raster: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/byGroup/C1-R_RL_postMEL_PRB+KLT_by_group.png
Saved grouped subject raster: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/byGroup/C2-L_X_postMEL_MW+KLT_by_group.png
Saved grouped subject raster: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/byGroup/C2-R_RL_postMEL_NK+KLT_by_group.png
Saved group comparison raster: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/Test/pre-processed/plots/byGroup/group_comparison_control_vs_experimental.png
Group-based raster plotting complete.
